# Notebook for PC 9 -- SENT specimen

## General imports

For development: If the `pygmsh_mesh_functions` module changes, the notebook does not need to be restarted completely.
Only the cells depending on the project module have to be re-executed. Useful for debugging.

In [1]:
%matplotlib nbagg

%load_ext autoreload

%autoreload 2

In [2]:
# Standard python libraries
import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt

from dolfin import *

## General user settings

In [3]:
# Choose model 'AT1' or 'AT2'
model = 'AT1'

# Activate crack or not
crack = False

#Choose material
material = 'verowhite'
#material = 'tangoblack'



# Activate Amor et al. split
split = False

#meshname = "mesh_tuned_2.xdmf"
meshname = "mesh_tuned.xdmf"

# Choose solver (set one or the other True)
#
newton_solver = False
snes_solver = True
snes_solver_lu = False


# Define output targets
out_dir = pathlib.Path("output_files")
out_dir.mkdir(parents=True, exist_ok=True)

# Define output name.
variant = "SENT_finite_strain_no_crack_TB" # CHANGE THIS IF YOU RUN DIFFERENT VARIANTS. Otherwise output will just be overwritten
outname = "{:s}.xdmf".format(variant)
outdata = "ForcevsDisp_{:s}.txt".format(variant)

# Print flag for notebook mainly (set = False to deactivate  |  True to activate)
iprint = True

## Import the project helper module

In [4]:
# this import allows to set module variables from outside via
# `pygmsh_mesh_functions.var_xyz = ...`
import pygmsh_mesh_functions

# convenience import: import all definitions
from pygmsh_mesh_functions import *

# Create the domain

Use the commands described in the file `project_II_meshing.pdf`.

In [5]:
# Define min and max mesh size. The max value should be less than the el value of the phase field
# If the following lines are commented an predefined range is used.

%reload_ext autoreload

meshsize_min = 0.0025
meshsize_max = 0.1

pygmsh_mesh_functions.hmeshmin = meshsize_min
pygmsh_mesh_functions.hmeshmax = meshsize_max

# Clean the pygmsh geometry from the pygmsh_mesh_functions module
reset_geometry()


# Create a rectangle by defining two corner points
Lx=1  # x length of the rectangle
Ly=1  # y length of the rectangle

# Set X0 (is the bottom left corner of the domain)
X0 = np.array([0, 0])

#
## Create pristine domain (the points should be given in a counterclockwise fashion)
domain = add_polygon([X0, (Lx, 0), (Lx, Ly), (0, Ly)])

In [6]:
# Add crack (notch)
if crack:
    d0 = 1e-5
    xc0 = Lx/4
    yc0 = Ly/2
    crack = add_crack([
        [0,   yc0 - d0/2],
        [xc0, yc0],
        [0,   yc0 + d0/2]
        ],
        mesh_size=meshsize_min
    )
    domain = subtract(domain, crack)
    

# Add holes
holes = False
if holes:
    XE1 = X0 + [2/3 * Lx, 2/3 * Ly] # center
    aE1 = Lx/5 # a (long semi-axis)
    bE1 = Lx/8 # b (short semi-axis)
    ellipse1 = add_ellipse(XE1, aE1, bE1, angle=np.pi/4,mesh_size=meshsize_min)
    XE2 = X0 + [4/5 * Lx, 1/3 * Ly] # center
    aE2 = Lx/8 # a (long semi-axis)
    bE2 = Ly/16 # b (short semi-axis)
    ellipse2 = add_ellipse(XE2, aE2, bE2, angle=0.0,mesh_size=meshsize_min)
    domain = subtract(domain, [ellipse1, ellipse2])   

In [7]:
# Create and load a FEniCS mesh through gmsh and meshio
mesh = create_fenics_mesh(verbose=True)

# Uncomment the line below to show the mesh
#mesh

Info    : Running '/Users/bhat/miniforge3/envs/xmeca2023/bin/gmsh -3 /var/folders/43/prmgkvk10p98dbz8d5th6hmm0000gn/T/tmplu7r64v0.geo -format msh -bin -o /var/folders/43/prmgkvk10p98dbz8d5th6hmm0000gn/T/tmpng6ysr2i.msh' [Gmsh 4.9.0, 1 node, max. 1 thread]
Info    : Started on Fri Mar  3 09:42:07 2023
Info    : Reading '/var/folders/43/prmgkvk10p98dbz8d5th6hmm0000gn/T/tmplu7r64v0.geo'...
Info    : Done reading '/var/folders/43/prmgkvk10p98dbz8d5th6hmm0000gn/T/tmplu7r64v0.geo'
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 50%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.000806125s, CPU 0.003617s)
Info    : Meshing 2D...
Info    : Meshing surface 6 (Plane, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.0468279s, CPU 0.17262s)
Info    : Meshing 3D...
Info    : Done meshing 3D (Wall 0.000247917s, CPU 0.000359s)
Info    : 1125 nodes 2252 elements
Info    : Writing '

/Users/bhat/miniforge3/envs/xmeca2023/lib/python3.8/site-packages/meshio/_mesh.py:130: UserWarning: prune() will soon be deprecated. Use remove_lower_dimensional_cells(), remove_orphaned_nodes() instead.
  warnings.warn(


In [8]:
# Compute the volume for evaluation of the "porosity target"
assemble(1*dx(domain=mesh))

0.9999999999999818

In [9]:
# Check the number of mesh nodes/vertices
len(mesh.coordinates())

1125

## Load selected domain

In [10]:
# read the mesh file
#with create_XDMFFile(meshname) as xf:
#    xf.read(mesh)

In [11]:
#mesh

# Loading parameters

- Define loading amplitudes $\gamma$ and $\delta$ 
- Define time increments $dt$ via number of steps

In [12]:
# Displacement amplitudes (Dirichlet loading conditions)
delta_amp = Constant(0.4) # displacement in y
gamma_amp = Constant(0.0) # displacement in x

# Load here can be identified with time t in [0,1]
t = Constant(0.0)
t_end = Constant(1.0)

# Time stepping parameters 
number_steps = Constant(1000)
dt = Constant(t_end / number_steps)

# Function space for the displacement field $\mathbf{u}$ and the phase-field $\varphi$

In compact notation, we define a function space with the mesh followed by arguments used for the specification of the underlying finite element.


Below, "CG" and "element_order" define the element type and order. We use a standard Lagrangian element type defined by the keyword "CG". The element order defines the interpolation shape function that will be used inside the element. A value "1" leads to a constant strain triangular element, while increasing the element order leads to higher-order polynomial approximations.

The displacement field is in fact a vector field and thus we call the "VectorFunctionSpace" command, which asks the element constructor to create a vector-valued element. W_u defines the vector space.

In [13]:
element_order=1

# Define (vector) function space for displacement (2 dof per node in 2D)
W_u = VectorFunctionSpace(mesh, 'CG', element_order)

# Define function space for phase field (1 dof per node)
W_phi = FunctionSpace(mesh, 'CG', element_order) 


if iprint: 
    print("Phi dofs:", W_phi.dim()) # the number of dofs
    print("Displacement dofs:",W_u.dim()) # the number of dofs
    print("Total dof:", W_u.dim()+W_phi.dim())

Phi dofs: 1125
Displacement dofs: 2250
Total dof: 3375


## FE Functions
 
The linear elasticity problem requires a single increment (inversion) to solve for the displacement field $\mathbf{u}$. Below we define current and previous values for dofs and history variables

In [14]:
u = Function(W_u, name="u")
u_n = u.copy(deepcopy=True); u_n.rename("u_n", "")

phi = Function(W_phi, name="phi")
phi_n = phi.copy(deepcopy=True); phi_n.rename("phi_n", "")

## Introduce the displacement variations

In FEniCS and FE language, the variations of the displacement fields take the name "test" and "trial" functions, corresponding to the first and second variations in the potential energy. For later use, we define the test and trial function for our solution spaces $\mathcal{C}_\mathbf{u}$. 
 
We use the convention "Delta_" $\rightarrow \Delta$ for trial and "delta_" $\rightarrow \delta$ for the test functions.

Trial and test functions for all dofs are defined below.

In [15]:
Delta_u, delta_u = TrialFunction(W_u), TestFunction(W_u)
Delta_phi, delta_phi = TrialFunction(W_phi), TestFunction(W_phi)

# Constitutive law

We focus on linear, elastic, isotropic constitutive laws. We present a version of the constitutive law, which uses the Young's modulus $E$ (MPa) and Poisson's ratio $\nu$ as material parameters 

## Material parameters

**Note**: "lambda" is a keyword in python and thus cannot be used as a variable name

In [16]:
# Define the Young's modulus and Poisson's ratio

if material == 'verowhite':
    E = Constant(1400e0)  # in MPa
    nu = Constant(0.42)
    sigc= Constant(150) # in MPa
    Gc = Constant(0.95)  # in N/mm = J /mm^2
elif material == 'tangoblack':
    E = Constant(3.4)  # in MPa
    nu = Constant(0.42)
    sigc= Constant(2.0) # in MPa
    Gc = Constant(0.1)  # in N/mm = J /mm^2

    
mu = E / (2 * (1.0 + nu))
lmbda = E * nu /((1 + nu) * (1 - 2.0 * nu))
elch=E*Gc/((1-nu*nu)*(sigc*sigc))


if model ==  'AT1':
    el=3*elch/8
elif model ==  'AT2':
    el=27*elch/256

    
#el = Constant(0.02)
eta = Constant(1.e-6)

# Note on usage of `Constant`s:
#  assign new value(s):  E.assign(3.0)
#  read value(s):        E.values(), E.values()[0], or, in case of scalars, float(E)
#  can be vector- and tensor-valued as well and in that case the size is not up to [0]

print('E=', float(E),',  nu=', float(nu))
print('mu=', float(mu),', lambda=', float(lmbda))
print('sig_c=',float(sigc),'  l_ch=', float(elch))
print('Gc=', float(Gc),',  epsilon=', float(el),',  eta=', float(eta))

E= 1400.0 ,  nu= 0.42
mu= 492.9577464788733 , lambda= 2588.028169014084
sig_c= 150.0   l_ch= 0.07177162592412714
Gc= 0.95 ,  epsilon= 0.026914359721547678 ,  eta= 1e-06


## Constitutive response functions


Here we will import some mathematical operations from the `ufl` package. In theory, all algebra from `ufl` should be available via "dolfin". However, due to API changes, some operators have been omitted. In case you are missing something in dolfin, we can look in `ufl`. Moreover, concerning constitutive models, the "ufl" version of operations are usually enough such that one can by default import them from `ufl`.

In [17]:
# IMPORTANT: DO NOT do "from ufl import *" because this overrides certain definitions 
# from `dolfin import *`
from ufl import (sym, grad, tr, Identity, dev, conditional, inner, transpose) 

Definition of the deformation gradient: $\boldsymbol{F} = \nabla (\mathbf{u} + \mathbf{X)}$ where $\mathbf{X}$ are referential coordinates.

In [18]:
d = u.ufl_shape[0] # space dimension
F = grad(u) + Identity(d) # symmetric gradient of argument u

Preliminary definitions to be used in the energy densities.

In [19]:
kappa = lmbda + 2 * mu / 2    # the last denominator is for 2D -> 2, 3D -> 3 


def det_F_compression(F):
    return conditional(det(F) < 1, det(F), 1)


def det_F_inflation(F):
    return conditional(det(F) > 1, det(F), 1)


def F_isochoric(F):
    return 1/det(F) * F


def g(phi, eta):
    return (1.0 - phi)**2 + eta


# model-dependent definitions
if model ==  'AT1':
    c_q = 2 / 3
    def q(phi):
        return phi
elif model == 'AT2':
    c_q = 1 / 2
    def q(phi):
        return phi**2

Definition of Neo-Hookean strain energy function (with $J=\mathrm{Det}\mathbf{F}$): $w(\mathbf{F})=\dfrac{\mu}{2} ({\mathbf{F}}:{\mathbf{F}} - d - 2\ln J) + \dfrac{\Lambda}{2} \,(J-1)^2$, the coupling term $g(\varphi)=(1-\varphi)^2+\eta_\epsilon$ and the damage energy $q(\varphi)=\varphi$ and their derivatives.

The following allows us to differentiate with respect to "F". It prevents direct expansion of "F" with respect to the primary variable "u" and will be used to evaluate the stresses.

In [20]:
# Make F a variable to prevent early symbolic expansion
F = variable(F)
J = det(F)

# elastic energy density with or without the split inspired by Amor et al. 2009
if split:
    raise NotImplementedError("Not implemented for finite strains")
else:
    wel = g(phi, eta) * (mu/2 * (inner(F, F) - 2 - 2 * ln(J)) + lmbda/2 * (J-1)**2)


wsurf =  Gc/(4*c_q) *(q(phi)/el+ el * grad(phi)**2.0)

# Enforcing bounds on phi (0 <= phi <= 1)
wpen = conditional(phi < 0,              # if
                   1e4 * E * phi**2,    # then 
                   conditional(phi > 1,  # else         if
                               1e6 * Gc * (phi-1)**2, # then
                               0))                    # else

# Enforcing irreversibility (phi >= phi_n)
wpen2 = conditional(phi < phi_n,               # if
                    1e4 * E * (phi-phi_n)**2, # then
                    0)                         #else

# Symbolic definition of the Cauchy stress
sigma = 1/J * diff(wel, F) * transpose(F)

In [21]:
# The following lines create FE functions for postprocessing. Only a limited set of FE spaces can
# be written to FEniCS xdmf files. Hence, we write the partially discontinuous stress and strain 
# to a fully discontinuous spaces, i.e. constant element (0th order element).
V_S_pp = TensorFunctionSpace(mesh, "DG", 0) 
sig_pp = Function(V_S_pp, name="sigma")
F_pp = Function(V_S_pp, name="F")

# For the energies, a continuous space is appropriate.
V_energy_pp = FunctionSpace(mesh, "CG", 1)
wel_pp = Function(V_energy_pp, name="elastic energy")
wsurf_pp = Function(V_energy_pp, name="surface energy")

# Define boundary conditions

The following few cells serve to apply the boundary conditions of the problem.

## Domain definitions

First, one needs to identify the boundary subdomains. The reference point `X0` (defined above) denotes the bottom left corner of the domain.

In [22]:
left = CompiledSubDomain("near(x[0], side) && on_boundary", side = X0[0])
right = CompiledSubDomain("near(x[0], side) && on_boundary", side = X0[0] + Lx)

bottom = CompiledSubDomain("near(x[1], side) && on_boundary", side = X0[1])
top = CompiledSubDomain("near(x[1], side) && on_boundary", side = X0[1] + Ly)

if crack:
    crack = CompiledSubDomain("between(x[0], {side, xc}) && near(x[1], yc, toly) && on_boundary", 
                          side = X0[0], xc=xc0, yc=yc0, toly=d0)

## Displacement boundary conditions

Considers uniaxial tension along direction $y$ at the top surface and full clamping at the bottom surface. At the top surface of the specimen, we apply a displacement amplitude $\mathbf{u}=0 \mathbf{e}_x+\delta\mathbf{e}_y$ and at the bottom $\mathbf{u}=\mathbf{0}$. The left and right sides are traction free. 
Hence, the Neumann boundary conditions are all zero, i.e., all tractions, whenever those are applied they are zero. 

In [23]:
# Actual Dirichlet boundary conditions. They are constructed with
#  - an FE space or a subspace (also for components of vectors)
#  - a value
#  - a SubDomain where the BC should be applied

# A helper element for spatially constant expressions.
# This is also useful for dolfin.Function in some cases.
const_element = FiniteElement("R", mesh.cell_name(), 0)


# Spatially constant expressions. We multiply the amplitude with the time
delta = Expression("t * delta_amp", t=t, delta_amp=delta_amp, element=const_element)
gamma = Expression("t * gamma_amp", t=t, gamma_amp=gamma_amp, element=const_element)


# Examples below for Dirichlet BCs
bc_bottom = DirichletBC(W_u.sub(1), Constant(0.0), bottom)
#bc_top_x = DirichletBC(W_u.sub(0), gamma, top)
bc_top_y = DirichletBC(W_u.sub(1), delta, top)
bc_pointwise = DirichletBC(W_u.sub(0), Constant(0.0), lambda x, on_boundary: near(x[0], 0.0) and near(x[1], 0.0), 
                           method="pointwise")


# Collect the BCs in a list
bc_u = [
    bc_pointwise,
    bc_bottom, 
#    bc_top_x,
    bc_top_y
]

## Phase-field boundary conditions

In [24]:
# phi = 1 at the initial crack
# More conditions must be added when there are additional crack domains
bc_phi = [
    DirichletBC(W_phi, Constant(0.0), bottom),
    DirichletBC(W_phi, Constant(0.0), top)
]

if crack:
    bc_phi.append(DirichletBC(W_phi, Constant(1.0), crack))

# Define the Total Potential Energy and its variations

The potential energy (without the split) of the system is defined as

$\mathcal{P}(\mathbf{u}, \varphi)= \int_{\Omega} g(\varphi) w(\boldsymbol{F}(\mathbf{u})) \textrm{d}\Omega + \dfrac{G_\texttt{c}}{4c_q}\int_{\Omega} \dfrac{q(\varphi)}{\epsilon} + \epsilon (\nabla \varphi)^2 \textrm{d}\Omega+\int_{\Omega} w_{pen}(\varphi)\textrm{d}\Omega$

For the split see above definitions

In [25]:
Pi = (wel + wsurf + wpen + wpen2) * dx  # dx = dV

Compute the force vector as the first variation of `Pi`, i.e. the directional derivative with respect to `u` in the direction of `delta_u`.

In [26]:
Force_u = derivative(Pi, u, delta_u)

Compute the Jacobian matrix (used for convergence of the Newton loops)
as the second variation of `Pi` or the first variation of `Force_u`
(directional derivative with respect to `u` in the direction of `Delta_u`).

In [27]:
Jac_u = derivative(Force_u, u, Delta_u)

Compute the force vector as the first variation of `Pi`
(directional derivative with respect to `phi` in the direction of `delta_phi`)

In [28]:
Force_phi = derivative(Pi, phi, delta_phi)

Compute the Jacobian matrix (used for convergence of the Newton loops) as the 
second variation of `Pi` or the first variation of `Force_phi`
(directional derivative with respect to `phi` in the direction of `Delta_phi`)

In [29]:
Jac_phi = derivative(Force_phi, phi, Delta_phi) 

## Reaction forces

For the evaluation of the specimen performance, we need to evaluated the reaction (traction/force) to the 
prescribed displacement at the top.

In [30]:
boundaries = MeshFunction("size_t", mesh, mesh.topology().dim() - 1)
boundaries.set_all(0)

# mark the top as surface 1
top.mark(boundaries,1)

# mark the right as surface 2
right.mark(boundaries,2)

This information can the be used to create a "Measure" for integration of (outer) boundaries.
Such a measure for boundary integrals is indicated by `ds`.
However, in the present example we simply need it for postprocessing *reaction forces or tractions*.

In [31]:
ds = ds(subdomain_data=boundaries)

For postprocessing the traction we define the traction vector: $\mathbf{t}=\boldsymbol{\sigma}\cdot\mathbf{n}$

In [32]:
n = FacetNormal(mesh)
traction = dot(sigma, n)

We will extract all values in the force displacement file but only one will make sense
given each time the boundary conditions.

In [33]:
total_force_x = traction[0]*ds(1) # The label ds(1) ---> refers to the Top surface (see above)
total_force_y = traction[1]*ds(1)

#total_force_x_right = traction[0]*ds(2) # The label ds(2) ---> refers to the Right surface (see above)
#total_force_y_right = traction[1]*ds(2)

Also, if we are pulling on the right surface, uncomment the above two last lines

# Solution procedure

In [34]:
# Some time increment parameters and staggered scheme tolerances
tol = 1e-4
dt0 = dt
dtmin = 1e-6
dtmax = dt

nitermax = 1000
flag_nitermax = 0

max_solver_iter = 20


write_freq = 1 # this controls the frequency at which we write in the results files
ifreq = 0


if newton_solver:
    # Solver parameters for Newton solver
    solver_parameters = {"nonlinear_solver": "newton",
                         "newton_solver": {"maximum_iterations": max_solver_iter,
                                           "linear_solver": "mumps", # a robust and quite fast direct solver
                                           "absolute_tolerance": tol,
                                           "relative_tolerance": tol}}
elif snes_solver:
    # Solver parameters for SNES solver
    solver_parameters = {"nonlinear_solver": "snes",
                         "snes_solver": {"line_search": "cp",
                                         "linear_solver": "cg",   
                                         "preconditioner": "amg",
                                         "maximum_iterations": max_solver_iter,
                                         "report": True,
                                         "error_on_nonconvergence": False,
                                         "absolute_tolerance": tol,
                                         "relative_tolerance": tol}}

elif snes_solver_lu:
    # Solver parameters for SNES solver
    solver_parameters = {"nonlinear_solver": "snes",
                          "snes_solver": {"line_search":"basic",
                                          "linear_solver": "lu",   
                                          "preconditioner": "amg",
                                          "maximum_iterations": max_solver_iter,
                                          "report": True,
                                          "error_on_nonconvergence": False}}

    
# NOTE: If the file already exists, it will not destroyed but data will simply be added.
# This can cause corrupted files, e.g., when the mesh has changed between two runs.
# Therefore, existing output files are removed
os.system("rm -f {!s}".format(out_dir / outname))

xf = create_XDMFFile(out_dir / outname)


with open(out_dir / outdata, 'w') as logf:
    # use some advanced string formatting
    logf.write(("{:>16s}"*5 + "\n").format("gamma", "delta", "Tx", "Ty", "niter"))


# Choose which forces to print out in the results file    
ftotx = assemble(total_force_x)
ftoty = assemble(total_force_y)

#ftotx = assemble(total_force_x_right)
#ftoty = assemble(total_force_y_right)


with open(out_dir / outdata, 'a') as logf:
    logf.write("{:16.6e}".format(float(0.0)))
    logf.write("{:16.6e}".format(float(0.0)))
    logf.write("{:16.6e}".format(ftotx))                
    logf.write("{:16.6e}".format(ftoty))   
    logf.write("{:16d}\n".format(0))

    
# reset vectors (in case of repeated runs for testing)
u.vector().zero()
u_n.vector().zero()
phi.vector().zero()
phi_n.vector().zero()

# backup, in case something goes wrong --> assign back to u and p_0 in case of failure
u_t = u.copy(deepcopy=True)
phi_t = phi.copy(deepcopy=True)
t_t = Constant(0.0)

# set time to zero
t.assign(0)

# write initial state
xf.write(u, float(t))
xf.write(phi, float(t))

# project strain, stress and energy densities for postprocessing
F_pp.assign(project(F, V_S_pp))
sig_pp.assign(project(sigma, V_S_pp))
wel_pp.assign(project(wel, V_energy_pp))
wsurf_pp.assign(project(wsurf, V_energy_pp))

xf.write(F_pp, float(t))
xf.write(sig_pp, float(t))
xf.write(wel_pp, float(t))
xf.write(wsurf_pp, float(t))



# The time loop
# =============
while float(t) + float(dt) < float(t_end) + 1e-6: # float(constant) works if the Constant is scalar
    
    t.assign(t + dt)

    
    
    niter = 0
    err = 1
    solver_error = False


#   Increase time increment if convergence is good: niter <= 3                    
#    if niter < 5 and deltaT < deltaT0: 
#        deltaT = 1.5 * deltaT


    while err > tol or np.isnan(err) or solver_error:
        niter += 1
        
        # Decrease time step if niter > nitermax
        if niter > nitermax or np.isnan(err):
            t.assign(t - dt)
            dt = 0.5 * dt
            niter = 1
            t.assign(t + dt)
            assign(u, u_t)
            assign(u_n, u_t)
            assign(phi, phi_t)
            assign(phi_n, phi_t)
            print('-o-o-o-o-o-o-o')
            print('Reducing time increment to dt=', float(dt))
        
        
        # Stopping criterion for minimum time step
        if float(dt) < float(dtmin):
            flag_nitermax +=1
            nitermax = 4*nitermax
            if nitermax > 1000:
                raise RuntimeError("Already reached minimum dt. Giving up.")

        # Solve the linear problems using a standard inversion technique
        solver_error = False
        try:
            solve(Force_u == 0, u, bcs=bc_u, J=Jac_u, solver_parameters=solver_parameters)
        except RuntimeError:
            solver_error = True
            
        err_u = errornorm(u, u_n, norm_type='l2', mesh=None)
        u_n.assign(u)
#        print ('======> error_u=', err_u)

        try:
            solve(Force_phi == 0, phi, bcs=bc_phi, J=Jac_phi, solver_parameters=solver_parameters)
        except RuntimeError:
            solver_error = True
        err_phi = errornorm(phi, phi_n, norm_type='l2', mesh=None)
        phi_n.assign(phi)
#        print ('======> error_phi=', err_phi)


        # Evaluate error norms 
        # dolfin.errornorm is a special evaluation function for small differences
        err = max(err_u, err_phi)
        print ('niter=', niter, ', max error=', err,', error_u=',err_u,', error_phi=',err_phi)
    

    if err <= tol and not solver_error:
        
        ifreq += 1
        
        
        print ('Iterations:', niter, ', Total time:', float(t),  ', Time Increment:', float(dt))


        ftotx = assemble(total_force_x)
        ftoty = assemble(total_force_y)

        #ftotx = assemble(total_force_x_right)
        #ftoty = assemble(total_force_y_right)

               
        # We write in the force-displacement file at every converged increment            
        with open(out_dir / outdata, 'a') as logf:
            logf.write("{:16.6e}".format(float(t*gamma_amp)))
            logf.write("{:16.6e}".format(float(t*delta_amp)))
            logf.write("{:16.6e}".format(ftotx))
            logf.write("{:16.6e}".format(ftoty))
            logf.write("{:16d}\n".format(niter))                
            
        # Always write the first increment irrespective of frequency writing (tol can be any small number << deltaT)
            
        if ifreq % write_freq == 0 or float(t) < (1+tol) * float(dt):
            
            ifreq = 0
                

            xf.write(u, float(t))
            xf.write(phi, float(t))

            
            # compute epsilon (strain), sigma (stress) and energy densities for postprocessing
            F_pp.assign(project(F, V_S_pp))
            sig_pp.assign(project(sigma, V_S_pp))
            wel_pp.assign(project(wel, V_energy_pp))
            wsurf_pp.assign(project(wsurf, V_energy_pp))

            # write them to a file
            xf.write(F_pp, float(t))
            xf.write(sig_pp, float(t))
            xf.write(wel_pp, float(t))
            xf.write(wsurf_pp, float(t))

            
        if niter < 5 and float(dt) < float(dtmax):
            dt = 2 * dt
            if float(dt) > float(dtmax):
                dt = dtmax
            print('-o-o-o-o-o-o-o')
            print('Increasing time increment to dt=', float(dt))



print('-->  Simulation completed')

Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Solving nonlinear variational problem.
  0 SNES Function norm 9.071746019412e+00 
  1 SNES Function norm 6.527209031277e-05 
  PETSc SNES solver converged in 1 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Calling FFC just-in-time (JIT) compiler, this may take some time.
Solving nonlinear variational prob

  0 SNES Function norm 1.386110636469e-04 
  1 SNES Function norm 4.680172571679e-11 
  PETSc SNES solver converged in 1 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
niter= 2 , max error= 8.138052839588314e-11 , error_u= 0.0 , error_phi= 8.138052839588314e-11
Iterations: 2 , Total time: 0.006 , Time Increment: 0.001
Solving nonlinear variational problem.
  0 SNES Function norm 9.043698947190e+00 
  1 SNES Function norm 6.536863349463e-05 
  PETSc SNES solver converged in 1 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
Solving nonlinear variational problem.
  0 SNES Function norm 3.707832362311e-05 
  PETSc SNES solver converged in 0 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
niter= 1 , m

niter= 1 , max error= 0.0002845957301231939 , error_u= 0.0002845957301231939 , error_phi= 2.396036125762917e-10
Solving nonlinear variational problem.
  0 SNES Function norm 6.544966350877e-05 
  PETSc SNES solver converged in 0 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
Solving nonlinear variational problem.
  0 SNES Function norm 6.559332314958e-11 
  PETSc SNES solver converged in 0 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
niter= 2 , max error= 0.0 , error_u= 0.0 , error_phi= 0.0
Iterations: 2 , Total time: 0.013000000000000005 , Time Increment: 0.001
Solving nonlinear variational problem.
  0 SNES Function norm 9.011115979962e+00 
  1 SNES Function norm 6.552256382054e-05 
  PETSc SNES solver converged in 1 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: De

niter= 1 , max error= 0.00028433977022898713 , error_u= 0.00028433977022898713 , error_phi= 3.536927246291187e-10
Solving nonlinear variational problem.
  0 SNES Function norm 6.535086030608e-05 
  PETSc SNES solver converged in 0 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
Solving nonlinear variational problem.
niter= 2 , max error= 0.0 , error_u= 0.0 , error_phi= 0.0
Iterations: 2 , Total time: 0.01900000000000001 , Time Increment: 0.001
  0 SNES Function norm 9.682609345864e-11 
  PETSc SNES solver converged in 0 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
Solving nonlinear variational problem.
  0 SNES Function norm 8.983339646262e+00 
  1 SNES Function norm 6.537072236859e-05 
  PETSc SNES solver converged in 1 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: D

niter= 1 , max error= 0.00028408515376288524 , error_u= 0.00028408515376288524 , error_phi= 4.671913504029874e-10
Solving nonlinear variational problem.
  0 SNES Function norm 6.586159658872e-05 
  PETSc SNES solver converged in 0 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
Solving nonlinear variational problem.
  0 SNES Function norm 1.278971766026e-10 
  PETSc SNES solver converged in 0 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
niter= 2 , max error= 0.0 , error_u= 0.0 , error_phi= 0.0
Iterations: 2 , Total time: 0.025000000000000015 , Time Increment: 0.001
Solving nonlinear variational problem.
  0 SNES Function norm 8.955703896163e+00 
  1 SNES Function norm 6.565807560097e-05 
  PETSc SNES solver converged in 1 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: 

...

*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
Solving nonlinear variational problem.
  0 SNES Function norm 3.779668475971e-01 
  1 SNES Function norm 2.442803882874e-01 
  2 SNES Function norm 8.200623210175e-03 
  3 SNES Function norm 3.125444374104e-03 
  4 SNES Function norm 1.870262799151e-03 
  5 SNES Function norm 1.761220607575e-04 
  6 SNES Function norm 3.701317634730e-05 
  PETSc SNES solver converged in 6 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
niter= 1 , max error= 0.0002964176534660687 , error_u= 0.0002964176534660687 , error_phi= 4.816252085314663e-07
Solving nonlinear variational problem.
  0 SNES Function norm 5.404505377894e-06 
  PETSc SNES solver converged in 0 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
Solving nonlin

  0 SNES Function norm 3.779725490001e-01 
  1 SNES Function norm 2.442889886979e-01 
  2 SNES Function norm 8.015145254670e-03 
  3 SNES Function norm 2.950068359662e-03 
  4 SNES Function norm 1.784652495534e-03 
  5 SNES Function norm 9.600565004581e-05 
  PETSc SNES solver converged in 5 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
niter= 2 , max error= 4.776113755483213e-07 , error_u= 0.0 , error_phi= 4.776113755483213e-07
Iterations: 2 , Total time: 0.7250000000000005 , Time Increment: 0.001
Solving nonlinear variational problem.
  0 SNES Function norm 9.064441001115e+00 
  1 SNES Function norm 9.217666847314e-07 
  PETSc SNES solver converged in 1 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
Solving nonlinear variational problem.
  0 SNES Function norm 3.779615456328e-01 
  1 SNES

Solving nonlinear variational problem.
  0 SNES Function norm 3.779572877354e-01 
  1 SNES Function norm 2.442687562593e-01 
  2 SNES Function norm 8.058413850210e-03 
  3 SNES Function norm 3.035138669163e-03 
  4 SNES Function norm 1.776016572213e-03 
  5 SNES Function norm 1.498792384556e-04 
  6 SNES Function norm 9.910808370990e-06 
  PETSc SNES solver converged in 6 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
niter= 1 , max error= 0.00029660501154649875 , error_u= 0.00029660501154649875 , error_phi= 4.793853659288686e-07
Solving nonlinear variational problem.
  0 SNES Function norm 6.440112124524e-06 
  PETSc SNES solver converged in 0 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
Solving nonlinear variational problem.
  0 SNES Function norm 3.779671827129e-01 
  1 SNES Function no

Solving nonlinear variational problem.
  0 SNES Function norm 3.779520280200e-01 
  1 SNES Function norm 2.442624021992e-01 
  2 SNES Function norm 7.981447758990e-03 
  3 SNES Function norm 2.975705259527e-03 
  4 SNES Function norm 1.695619999082e-03 
  5 SNES Function norm 1.415689925094e-04 
  6 SNES Function norm 1.126439860319e-05 
  PETSc SNES solver converged in 6 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
niter= 1 , max error= 0.00029670796906370824 , error_u= 0.00029670796906370824 , error_phi= 4.782369316902914e-07
Solving nonlinear variational problem.
  0 SNES Function norm 6.348625939652e-06 
  PETSc SNES solver converged in 0 iterations with convergence reason CONVERGED_FNORM_ABS.
*** Warning: Degree of exact solution may be inadequate for accurate result in errornorm.
Solving nonlinear variational problem.
  0 SNES Function norm 3.779618820317e-01 
  1 SNES Function no

KeyboardInterrupt: 

# Analyze the results

In [ ]:
data = np.loadtxt(out_dir / outdata, skiprows=1)
gamma_array, delta_array, fx, fy = data[:,0], data[:,1], data[:,2], data[:,3]

In [ ]:
work = sum((gamma_array[1:] - gamma_array[:-1]) * (fx[1:] + fx[:-1])/2) + \
       sum((delta_array[1:] - delta_array[:-1]) * (fy[1:] + fy[:-1])/2)
work

In [ ]:
plt.plot(delta_array, fy)

## Students-Q: What do you think about the plot above?